In [ ]:
#@title 1 · 在 A100 / 2026.10 准备冻结输入与 CPU 预检
import os, re, subprocess, sys, tempfile
from pathlib import Path
from google.colab import userdata

for key in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN", "HUGGINGFACE_HUB_TOKEN", "GH_TOKEN", "GITHUB_TOKEN", "WANDB_API_KEY"):
    os.environ.pop(key, None)
pin = (os.environ.get("FI_COMMIT") or input("粘贴验收回执中的 40 位 Git Commit SHA：")).strip()
if not re.fullmatch(r"[0-9a-f]{40}", pin):
    raise ValueError("需要完整的 40 位冻结 SHA")
repo = Path(tempfile.mkdtemp(prefix="fi-code-", dir="/content"))
def run(*args):
    subprocess.run(args, cwd=repo, check=True)
run("git", "init", "--quiet")
run("git", "fetch", "--quiet", "--depth=1", "https://github.com/rrrrrredy/fault-injector.git", pin)
run("git", "checkout", "--quiet", "--detach", "FETCH_HEAD")
if subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip() != pin:
    raise RuntimeError("检出版本与冻结 SHA 不符")
os.chdir(repo)
sys.path.insert(0, str(repo))
from stage2.colab_assets import prepare
work = Path("/content/work_fi")
smoke_done = calibration_done = False
python = prepare(work, lambda: userdata.get("HF_TOKEN"))
run(python, "-m", "stage2.colab_grpo", "--mode", "mock-preflight")

In [ ]:
#@title 2 · Qwen3-4B 走 30 步，首个有效候选走 Factory 与 RunCase
smoke_done = False
run(python, "-m", "stage2.colab_grpo", "--mode", "smoke-4b")
smoke_done = True

In [ ]:
#@title 3 · Qwen3-14B 独立进程标定实际 16K 长度
calibration_done = False
run(python, "-m", "stage2.colab_grpo", "--mode", "calibrate-14b")
calibration_done = True

In [ ]:
#@title 4 · 两个进程退出后归档，挂载 Drive，读回校验
import json
from google.colab import drive
from stage2.colab_assets import collect_outputs, copy_and_verify

if not (smoke_done and calibration_done):
    raise RuntimeError("请先完成本次会话的两个 GPU 作业")
files = collect_outputs(work)
drive.mount("/content/drive")
destination = Path("/content/drive/MyDrive/fault_injector_2026/stage2_smoke") / (pin[:12] + "-" + files[0].parent.name)
restored = copy_and_verify(files, destination)
print(json.dumps({"directory": str(destination), "restored": restored}, indent=2))
print("4B Case 验收", json.loads((work / "smoke-4b_receipt.json").read_text())["case_status"])